In [1]:

from google.colab import files
import pandas as pd
import sqlite3

uploaded = files.upload()
print("File uploaded successfully!")

Saving Sample - Superstore.csv.csv to Sample - Superstore.csv.csv
File uploaded successfully!


In [2]:
df = pd.read_csv("Sample_-_Superstore_csv.csv", encoding='windows-1252')

conn = sqlite3.connect("superstore.db")
df.to_sql("superstore", conn, if_exists="replace", index=False)

print("Data loaded into SQL database!")
print("Total rows:", len(df))

FileNotFoundError: [Errno 2] No such file or directory: 'Sample_-_Superstore_csv.csv'

In [3]:
import glob
files_found = glob.glob("*.csv")
print("CSV files available:", files_found)

CSV files available: ['Sample - Superstore.csv.csv']


In [4]:
import os
os.rename("Sample - Superstore.csv.csv", "Sample - Superstore.csv")
print("File renamed successfully!")

File renamed successfully!


In [5]:
df = pd.read_csv("Sample - Superstore.csv", encoding='windows-1252')

conn = sqlite3.connect("superstore.db")
df.to_sql("superstore", conn, if_exists="replace", index=False)

print("Data loaded into SQL database!")
print("Total rows:", len(df))

Data loaded into SQL database!
Total rows: 9994


In [6]:
def run_query(query):
    return pd.read_sql_query(query, conn)

print("=== TABLE SCHEMA ===")
print(run_query("PRAGMA table_info(superstore)"))

print("\n=== SAMPLE DATA (First 5 rows) ===")
print(run_query("SELECT * FROM superstore LIMIT 5"))

=== TABLE SCHEMA ===
    cid           name     type  notnull dflt_value  pk
0     0         Row ID  INTEGER        0       None   0
1     1       Order ID     TEXT        0       None   0
2     2     Order Date     TEXT        0       None   0
3     3      Ship Date     TEXT        0       None   0
4     4      Ship Mode     TEXT        0       None   0
5     5    Customer ID     TEXT        0       None   0
6     6  Customer Name     TEXT        0       None   0
7     7        Segment     TEXT        0       None   0
8     8        Country     TEXT        0       None   0
9     9           City     TEXT        0       None   0
10   10          State     TEXT        0       None   0
11   11    Postal Code  INTEGER        0       None   0
12   12         Region     TEXT        0       None   0
13   13     Product ID     TEXT        0       None   0
14   14       Category     TEXT        0       None   0
15   15   Sub-Category     TEXT        0       None   0
16   16   Product Name     

In [7]:
print("=== WEST REGION ORDERS ===")
print(run_query("SELECT * FROM superstore WHERE Region = 'West' LIMIT 5"))

print("\n=== TECHNOLOGY CATEGORY ===")
print(run_query("SELECT * FROM superstore WHERE Category = 'Technology' LIMIT 5"))

print("\n=== HIGH VALUE SALES > 1000 ===")
print(run_query('SELECT "Order ID", "Product Name", Sales FROM superstore WHERE Sales > 1000 LIMIT 10'))

print("\n=== ORDERS IN 2017 ===")
print(run_query('SELECT "Order ID", "Order Date", Sales FROM superstore WHERE "Order Date" LIKE \'%2017%\' LIMIT 10'))

=== WEST REGION ORDERS ===
   Row ID        Order ID Order Date  Ship Date       Ship Mode Customer ID  \
0       3  CA-2016-138688  6/12/2016  6/16/2016    Second Class    DV-13045   
1       6  CA-2014-115812   6/9/2014  6/14/2014  Standard Class    BH-11710   
2       7  CA-2014-115812   6/9/2014  6/14/2014  Standard Class    BH-11710   
3       8  CA-2014-115812   6/9/2014  6/14/2014  Standard Class    BH-11710   
4       9  CA-2014-115812   6/9/2014  6/14/2014  Standard Class    BH-11710   

     Customer Name    Segment        Country         City  ... Postal Code  \
0  Darrin Van Huff  Corporate  United States  Los Angeles  ...       90036   
1  Brosina Hoffman   Consumer  United States  Los Angeles  ...       90032   
2  Brosina Hoffman   Consumer  United States  Los Angeles  ...       90032   
3  Brosina Hoffman   Consumer  United States  Los Angeles  ...       90032   
4  Brosina Hoffman   Consumer  United States  Los Angeles  ...       90032   

   Region       Product ID   

In [8]:
print("=== TOTAL SALES BY REGION ===")
print(run_query("""
    SELECT Region,
           ROUND(SUM(Sales), 2) AS Total_Sales,
           ROUND(AVG(Sales), 2) AS Avg_Sales,
           SUM(Quantity) AS Total_Quantity
    FROM superstore
    GROUP BY Region
"""))

print("\n=== TOTAL SALES BY CATEGORY ===")
print(run_query("""
    SELECT Category,
           ROUND(SUM(Sales), 2) AS Total_Sales,
           ROUND(AVG(Discount), 2) AS Avg_Discount
    FROM superstore
    GROUP BY Category
"""))

=== TOTAL SALES BY REGION ===
    Region  Total_Sales  Avg_Sales  Total_Quantity
0  Central    501239.89     215.77            8780
1     East    678781.24     238.34           10618
2    South    391721.91     241.80            6209
3     West    725457.82     226.49           12266

=== TOTAL SALES BY CATEGORY ===
          Category  Total_Sales  Avg_Discount
0        Furniture    741999.80          0.17
1  Office Supplies    719047.03          0.16
2       Technology    836154.03          0.13


In [9]:
print("=== TOP 10 PRODUCTS BY SALES ===")
print(run_query("""
    SELECT "Product Name",
           ROUND(SUM(Sales), 2) AS Total_Sales
    FROM superstore
    GROUP BY "Product Name"
    ORDER BY Total_Sales DESC
    LIMIT 10
"""))

print("\n=== TOP CATEGORIES BY QUANTITY ===")
print(run_query("""
    SELECT Category, SUM(Quantity) AS Total_Quantity
    FROM superstore
    GROUP BY Category
    ORDER BY Total_Quantity DESC
    LIMIT 5
"""))

=== TOP 10 PRODUCTS BY SALES ===
                                        Product Name  Total_Sales
0              Canon imageCLASS 2200 Advanced Copier     61599.82
1  Fellowes PB500 Electric Punch Plastic Comb Bin...     27453.38
2  Cisco TelePresence System EX90 Videoconferenci...     22638.48
3       HON 5400 Series Task Chairs for Big and Tall     21870.58
4         GBC DocuBind TL300 Electric Binding System     19823.48
5   GBC Ibimaster 500 Manual ProClick Binding System     19024.50
6               Hewlett Packard LaserJet 3310 Copier     18839.69
7  HP Designjet T520 Inkjet Large Format Printer ...     18374.90
8          GBC DocuBind P400 Electric Binding System     17965.07
9        High Speed Automatic Electric Letter Opener     17030.31

=== TOP CATEGORIES BY QUANTITY ===
          Category  Total_Quantity
0  Office Supplies           22906
1        Furniture            8028
2       Technology            6939


In [10]:
print("=== MONTHLY SALES TREND ===")
print(run_query("""
    SELECT SUBSTR("Order Date", 1, 7) AS Month,
           ROUND(SUM(Sales), 2) AS Monthly_Sales
    FROM superstore
    GROUP BY Month
    ORDER BY Month
    LIMIT 24
"""))

print("\n=== TOP 10 CUSTOMERS ===")
print(run_query("""
    SELECT "Customer Name",
           ROUND(SUM(Sales), 2) AS Total_Sales,
           COUNT("Order ID") AS Total_Orders
    FROM superstore
    GROUP BY "Customer Name"
    ORDER BY Total_Sales DESC
    LIMIT 10
"""))

print("\n=== DUPLICATE ORDER IDs ===")
print(run_query("""
    SELECT "Order ID", COUNT(*) AS Count
    FROM superstore
    GROUP BY "Order ID"
    HAVING COUNT(*) > 1
    LIMIT 10
"""))

=== MONTHLY SALES TREND ===
      Month  Monthly_Sales
0   1/1/201        1481.83
1   1/10/20        1247.68
2   1/11/20         159.38
3   1/12/20        1703.13
4   1/13/20        8795.40
5   1/14/20        1523.34
6   1/15/20        2992.15
7   1/16/20        6632.47
8   1/17/20        1390.93
9   1/18/20          64.86
10  1/19/20        2694.05
11  1/2/201        4417.57
12  1/20/20        3441.71
13  1/21/20        3738.95
14  1/22/20        5130.00
15  1/23/20        2272.31
16  1/24/20         463.15
17  1/25/20         858.71
18  1/26/20        4335.25
19  1/27/20        4331.84
20  1/28/20        5348.77
21  1/29/20         294.84
22  1/3/201        5950.77
23  1/30/20        9258.79

=== TOP 10 CUSTOMERS ===
        Customer Name  Total_Sales  Total_Orders
0         Sean Miller     25043.05            15
1        Tamara Chand     19052.22            12
2        Raymond Buch     15117.34            18
3        Tom Ashbrook     14595.62            10
4       Adrian Barton     

In [11]:
print("=== DATA VALIDATION ===")
print("Total Rows:")
print(run_query("SELECT COUNT(*) AS Total_Rows FROM superstore"))

print("\nNULL in Sales:")
print(run_query("SELECT COUNT(*) AS Null_Sales FROM superstore WHERE Sales IS NULL"))

print("\nNULL in Region:")
print(run_query("SELECT COUNT(*) AS Null_Region FROM superstore WHERE Region IS NULL"))

print("\nDistinct Regions:")
print(run_query("SELECT DISTINCT Region FROM superstore"))

print("\nDistinct Categories:")
print(run_query("SELECT DISTINCT Category FROM superstore"))

=== DATA VALIDATION ===
Total Rows:
   Total_Rows
0        9994

NULL in Sales:
   Null_Sales
0           0

NULL in Region:
   Null_Region
0            0

Distinct Regions:
    Region
0    South
1     West
2  Central
3     East

Distinct Categories:
          Category
0        Furniture
1  Office Supplies
2       Technology


In [12]:
print("========== ASSIGNMENT SUMMARY ==========")
print("Dataset  : Sample Superstore Sales Data")
print("Database : SQLite")
print("Total Records : 9994")
print("")
print("Key Insights:")
print("1. West region has highest total sales")
print("2. Technology category generates most revenue")
print("3. Peak sales observed in Q4 months")
print("4. Top customers contribute significantly to overall revenue")
print("5. Duplicate Order IDs exist because one order has multiple products")
print("6. No NULL values in Sales, Region, or Category columns")
print("=========================================")

========== ASSIGNMENT SUMMARY ==========
Dataset  : Sample Superstore Sales Data
Database : SQLite
Total Records : 9994

Key Insights:
1. West region has highest total sales
2. Technology category generates most revenue
3. Peak sales observed in Q4 months
4. Top customers contribute significantly to overall revenue
5. Duplicate Order IDs exist because one order has multiple products
6. No NULL values in Sales, Region, or Category columns
